# IFC geometry exploration — Infra-Rail.ifc

This notebook works through everything the IfcOpenShell Python docs
recommend for **exploring geometry** in an IFC model, applied to one real
file: `Infra-Rail.ifc` (a buildingSMART IFC4X3 rail infrastructure sample —
75 elements: rails, track elements, ballast courses, an element assembly,
and a proxy).

| # | Topic | Doc source |
|---|---|---|
| 1 | Individual processing (`create_shape`) | `docs/ifcopenshell-python/geometry_processing.rst` — "Individual processing" |
| 2 | Geometry iterator (bulk + `include` filter) | same doc — "Geometry iterator"; `docs/ifcopenshell/geometry_iterator.rst` |
| 3 | Manual parsing (raw entity graph, no `create_shape`) | same doc — "Manual parsing" |
| 4 | Serialisation (glTF/glb, OBJ) | same doc — "Geometry serialisation" |
| 5 | Geometry tree (clash detection, spatial queries, ray casting) | `docs/ifcopenshell-python/geometry_tree.rst` |

Sections 1, 2, and 4 started from three code samples given as a starting
point; section 3 and 5 extend into the rest of what those doc pages cover,
since "everything the docs recommend" is broader than those three snippets.

**A running theme:** the docs were written against a slightly different API
surface than what's installed here (`ifcopenshell` 0.8.5). Every place that
happened is called out inline, with what actually works instead — see the
summary at the end for the full list.


## Setup

In [1]:
import multiprocessing
import time
from collections import Counter
from pathlib import Path
from typing import cast

import ifcopenshell
import ifcopenshell.geom
import ifcopenshell.util.shape
import ifcopenshell.util.unit
from ifcopenshell import ifcopenshell_wrapper

IFC_PATH = (
    r"C:\Users\ReDI\Documents\GitHub\2026_GREAM\project\local_ifcopenshell"
    r"\IfcOpenShell_Explorations\ifc_analysis_portfolio\building_smart-samples\Infra-Rail.ifc"
)

model = ifcopenshell.open(IFC_PATH)
print(f"opened {Path(IFC_PATH).name}  ({Path(IFC_PATH).stat().st_size / 1024:.0f} KB, schema={model.schema})")


opened Infra-Rail.ifc  (238 KB, schema=IFC4X3)


## 1 · Individual processing — `create_shape()`

The simplest way to turn one IFC element into a mesh: vertices, edges,
faces, its placement matrix, and the materials/styles applied to it.

> "This section describes individual processing only. This is useful for
> learning how geometry processing works, but is not recommended for
> practical applications." — docs

For bulk work, use the iterator (section 2) instead.


In [2]:
def style_summary(style) -> str:
    """Best-effort description of an ifcopenshell.geom style/material object.

    The attribute surface the docs describe (`style.original_name()`,
    `style.has_diffuse`, `style.has_transparency` as a bool) does not match
    what this installed build actually exposes on the SWIG `style` wrapper:
    there is no `original_name()` / `has_diffuse`, and `has_transparency`
    is a *method*, not a property. `diffuse` is always present as a
    `colour` object with `.r()`/`.g()`/`.b()` rather than a plain tuple.
    Found by introspecting `dir(style)` once the documented attributes
    raised AttributeError.
    """
    bits = [f"name={style.name!r}"]
    diffuse = getattr(style, "diffuse", None)
    if diffuse is not None:
        bits.append(f"diffuse=({diffuse.r():.3f}, {diffuse.g():.3f}, {diffuse.b():.3f})")
    has_transparency = getattr(style, "has_transparency", None)
    if callable(has_transparency) and has_transparency():
        bits.append(f"transparency={style.transparency:.3f}")
    return ", ".join(bits)


element = model.by_type("IfcRail")[0]
print(f"Processing {element.is_a()} {element.GlobalId!r} {element.Name!r}\n")

settings = ifcopenshell.geom.settings()

# "Choosing a geometry kernel has a big impact on speed and capability. It
# is recommended to use the hybrid-cgal-simple-opencascade kernel." — docs
#
# create_shape()'s declared return type is a broad Union covering every
# possible call shape, since the function has no @overload variants to
# narrow it per-argument. Passing a whole IfcProduct with no `repr` and
# default (triangulating) settings always returns a TriangulationElement
# at runtime -- this cast just tells a type checker what we already know.
shape = cast(
    ifcopenshell_wrapper.TriangulationElement,
    ifcopenshell.geom.create_shape(settings, element, geometry_library="hybrid-cgal-simple-opencascade"),
)

print("guid           ", shape.guid)
print("id             ", shape.id)
print("resolved elem  ", model.by_guid(shape.guid))

# A unique geometry ID: IfcShapeRepresentation.id{-layerset-N}{-material-N}
# {-openings-[...]}{-world-coords} -- useful for caching/reuse.
print("geometry id    ", shape.geometry.id)


Processing IfcRail '3Ixq0AlaX2PR0EdMfWL49i' 'rail'

guid            3Ixq0AlaX2PR0EdMfWL49i
id              61
resolved elem   #61=IfcRail('3Ixq0AlaX2PR0EdMfWL49i',#1,'rail','Strong rails, guiding trains safely along their path.','rail',#67,#77,'454425.1027894.2183506.2042690.955698',$)
geometry id     76


In [3]:
# 4x4 placement matrix: first 3 columns are the local X/Y/Z axes
# (right-handed, unscaled), last column is the XYZ position.
matrix = ifcopenshell.util.shape.get_shape_matrix(shape)
location = matrix[:, 3][0:3]
print("location (xyz) ", location)

verts = shape.geometry.verts   # flat [x,y,z, x,y,z, ...]
edges = shape.geometry.edges   # flat vertex-index pairs; not necessarily triangulated
faces = shape.geometry.faces   # flat vertex-index triples; always triangulated
print(f"\n{len(verts) // 3} vertices, {len(edges) // 2} edges, {len(faces) // 3} triangles")

# Same data, grouped into nested arrays -- usually nicer to work with
# (numpy math, exporting to another format) than the flattened lists above.
grouped_verts = ifcopenshell.util.shape.get_vertices(shape.geometry)
grouped_faces = ifcopenshell.util.shape.get_faces(shape.geometry)
print("first 3 grouped verts:", grouped_verts[:3].tolist())
print("first 3 grouped faces:", grouped_faces[:3].tolist())

print("\nstyles (materials) applied to this shape:")
for style in shape.geometry.materials:
    print(" ", style_summary(style))

# Per-triangle bookkeeping: which style and which source IFC representation
# item produced each face. Same length as the triangle count.
material_ids = shape.geometry.material_ids
item_ids = shape.geometry.item_ids
print(f"\nmaterial_ids sample: {list(material_ids[:10])}")
print(f"item_ids sample:     {list(item_ids[:10])}")


location (xyz)  [ 8.66025405 55.00000001  7.60258212]

24 vertices, 65 edges, 44 triangles
first 3 grouped verts: [[-0.7139999999999993, -1.4438228390645235e-15, 0.12099999999999991], [-0.7417500000000002, -1.4438228390645235e-15, 0.12099999999999991], [-0.7139999999999935, 1.4438228390645235e-15, 0.17199999999999988]]
first 3 grouped faces: [[0, 1, 2], [3, 2, 1], [4, 2, 3]]

styles (materials) applied to this shape:
  name='IfcSurfaceStyleRendering-62', diffuse=(0.471, 0.314, 0.157), transparency=0.000

material_ids sample: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
item_ids sample:     [72, 72, 72, 72, 72, 72, 72, 72, 72, 72]


## 2 · Geometry iterator — bulk processing

> "For any bulk geometry processing, it is always recommended to use the
> iterator" over calling `create_shape()` in a loop — it's multi-threaded
> and caches/reuses identical geometry (e.g. Infra-Rail.ifc's 66
> `IfcTrackElement`s are likely repeats of a handful of distinct shapes).


In [4]:
settings = ifcopenshell.geom.settings()

t0 = time.time()
iterator = ifcopenshell.geom.iterator(
    settings, model, multiprocessing.cpu_count(), geometry_library="hybrid-cgal-simple-opencascade"
)

counts = Counter()
verts_total = 0
faces_total = 0

if iterator.initialize():
    while True:
        shape = iterator.get()
        element = model.by_id(shape.id)
        counts[element.is_a()] += 1
        verts_total += len(shape.geometry.verts) // 3
        faces_total += len(shape.geometry.faces) // 3
        # matrix / edges / materials / material_ids are all available here
        # too -- exactly what create_shape() gives you per element, just
        # driven across the whole model, multi-threaded.
        if not iterator.next():
            break

elapsed = time.time() - t0
n_elements = len(model.by_type("IfcElement"))
n_shaped = sum(counts.values())

print(f"{n_shaped}/{n_elements} elements produced geometry in {elapsed:.2f}s "
      f"({n_elements - n_shaped} have no representation)\n")
print(f"{verts_total} total vertices, {faces_total} total triangles\n")
print("by type:")
for ifc_class, n in counts.most_common():
    print(f"  {ifc_class:<24} {n}")


73/75 elements produced geometry in 0.06s (2 have no representation)

1216 total vertices, 2120 total triangles

by type:
  IfcTrackElement          66
  IfcRail                  4
  IfcCourse                2
  IfcBuildingElementProxy  1


In [5]:
# The `include` setting -- process only a subset of elements.
# "One of the more common settings used is the include setting, which
# specifies only to process certain geometry." -- docs
rails = model.by_type("IfcRail")
print(f"filtered pass: only {len(rails)} IfcRail element(s)")

filtered_iterator = ifcopenshell.geom.iterator(
    settings, model, multiprocessing.cpu_count(),
    include=rails, geometry_library="hybrid-cgal-simple-opencascade",
)
n = 0
if filtered_iterator.initialize():
    while True:
        filtered_iterator.get()
        n += 1
        if not filtered_iterator.next():
            break
print(f"  iterator yielded {n} shape(s) -- matches len(rails)={len(rails)}: {n == len(rails)}")


filtered pass: only 4 IfcRail element(s)
  iterator yielded 4 shape(s) -- matches len(rails)=4: True


## 3 · Manual parsing — reading the entity graph directly

> "IfcOpenShell lets you traverse any IFC entity graph [...] This approach
> requires an in-depth understanding of IFC geometry representations [...]
> but can be very simple and extremely fast to extract specific types of
> geometry [...] generally not recommended except in specific tasks." — docs

The docs' own example reads an `IfcCircle`'s `Radius` directly. Infra-Rail.ifc
doesn't use swept circular profiles — every element's Body representation is
an `IfcTriangulatedFaceSet` (a raw mesh) — so this adapts the same idea:
pull `Coordinates`/`CoordIndex` straight off the entity, no `create_shape()`
involved at all.

Two gotchas `create_shape()` normally hides from you:
- `CoordIndex` is **1-based**, not 0-based.
- Coordinates are in the file's length unit (millimeters here), not
  meters — use `ifcopenshell.util.unit` to find the scale factor.


In [6]:
unit_scale = ifcopenshell.util.unit.calculate_unit_scale(model)
print(f"unit_scale = {unit_scale} (1 file unit = {unit_scale} m)\n")

tessellations = model.by_type("IfcTriangulatedFaceSet")
print(f"{len(tessellations)} IfcTriangulatedFaceSet representation items in this file\n")

for tessellation in tessellations[:3]:
    coords = tessellation.Coordinates.CoordList  # tuple of (x, y, z), file units
    face_indices = tessellation.CoordIndex        # tuple of (i, j, k), 1-based

    # Which element owns this representation item? Walk back up the graph:
    # Item -> IfcShapeRepresentation -> IfcProductDefinitionShape -> element.
    owners = [
        el for el in model.by_type("IfcElement")
        if el.Representation
        and any(tessellation in rep.Items for rep in el.Representation.Representations)
    ]
    owner = owners[0] if owners else None

    print(f"item #{tessellation.id()}  owner={owner.is_a() if owner else '?'} "
          f"{owner.GlobalId if owner else ''}")
    print(f"  {len(coords)} raw coords, {len(face_indices)} triangles")

    # First triangle: 1-based file indices -> 0-based Python, mm -> meters.
    i, j, k = face_indices[0]
    triangle_m = [tuple(c * unit_scale for c in coords[idx - 1]) for idx in (i, j, k)]
    print(f"  first triangle (meters): {triangle_m}\n")


unit_scale = 0.001 (1 file unit = 0.001 m)

6 IfcTriangulatedFaceSet representation items in this file

item #72  owner=IfcRail 3Ixq0AlaX2PR0EdMfWL49i
  72 raw coords, 44 triangles
  first triangle (meters): [(-0.7139999999999993, -1.4438228390645235e-15, 0.12099999999999991), (-0.7417500000000002, -1.4438228390645235e-15, 0.12099999999999991), (-0.7139999999999935, 1.4438228390645235e-15, 0.17199999999999988)]

item #84  owner=IfcRail 0gOEB3nvP8w8b2x7dkCurB
  72 raw coords, 44 triangles
  first triangle (meters): [(-0.7139999999999935, -1.4438228390645235e-15, 0.17199999999999988), (-0.7417500000000002, 1.4438228390645235e-15, 0.12099999999999991), (-0.7139999999999993, 1.4438228390645235e-15, 0.12099999999999991)]

item #101  owner=IfcCourse 2VSzSEJibEmQCf1lSBr3vL
  252 raw coords, 164 triangles
  first triangle (meters): [(-1.5727050272165939, -4.33146851719357e-15, -0.1933285725060929), (-1.7500000000000582, 19.999999999999986, -0.31159867738694436), (-1.7500000000000815, 1.7325874

## 4 · Serialisation — glTF/glb and OBJ

Same shape as the sample this was built from: settings → serialiser →
iterator → write per shape → finalize. Both formats are produced here (the
docs show them as alternatives, one commented out) so they can be diffed
against each other.


In [7]:
out_dir = Path.cwd() / "output"
out_dir.mkdir(exist_ok=True)

settings = ifcopenshell.geom.settings()
settings.set("dimensionality", ifcopenshell.ifcopenshell_wrapper.CURVES_SURFACES_AND_SOLIDS)
# "Applying default materials is required in glTF serialisation." -- docs
settings.set("apply-default-materials", True)

serialiser_settings = ifcopenshell.geom.serializer_settings()
# Optional, but useful to uniquely identify objects in non-semantic formats.
serialiser_settings.set("use-element-guids", True)

# --- glTF / glb ---
glb_path = out_dir / "Infra-Rail.glb"
serialiser = ifcopenshell.geom.serializers.gltf(str(glb_path), settings, serialiser_settings)
serialiser.setFile(model)
serialiser.setUnitNameAndMagnitude("METER", 1.0)
serialiser.writeHeader()

iterator = ifcopenshell.geom.iterator(settings, model, multiprocessing.cpu_count())
n = 0
if iterator.initialize():
    while True:
        serialiser.write(iterator.get())
        n += 1
        if not iterator.next():
            break
serialiser.finalize()
print(f"wrote {glb_path.name}  ({glb_path.stat().st_size / 1024:.0f} KB, {n} shapes)")


wrote Infra-Rail.glb  (92 KB, 73 shapes)


In [8]:
# --- OBJ ---
# OBJ has no material/GUID metadata channel like glTF, so world coordinates
# are usually what you want (otherwise every element sits at its own local
# origin, which looks like a pile of overlapping geometry in a viewer).
settings.set("use-world-coords", True)

obj_path = out_dir / "Infra-Rail.obj"
mtl_path = out_dir / "Infra-Rail.mtl"
serialiser = ifcopenshell.geom.serializers.obj(str(obj_path), str(mtl_path), settings, serialiser_settings)
serialiser.setFile(model)
serialiser.setUnitNameAndMagnitude("METER", 1.0)
serialiser.writeHeader()

iterator = ifcopenshell.geom.iterator(settings, model, multiprocessing.cpu_count())
n = 0
if iterator.initialize():
    while True:
        serialiser.write(iterator.get())
        n += 1
        if not iterator.next():
            break
serialiser.finalize()
print(f"wrote {obj_path.name}  ({obj_path.stat().st_size / 1024:.0f} KB, {n} shapes)")


wrote Infra-Rail.obj  (99 KB, 73 shapes)


## 5 · Geometry tree — clash detection & spatial queries

Not in the original three samples, but the other major "geometry
exploration" capability the docs cover: building a tree from the model,
then using it to (a) clash one group of elements against another and
(b) select elements by point/box/ray.

**API note found while building this:** the docs' tree examples build the
tree by adding shapes one at a time from the iterator
(`tree.add_element(iterator.get())`). That works fine for
`clash_intersection_many`/`clash_collision_many`/`clash_clearance_many` and
for the *precise* `select()`/`select_ray()` queries below — but
`select_box()` (querying by an element or a raw point) returned an empty
list for **every** query on this build, including an element querying its
own bounding box, which the docs promise always returns at least itself.
Building the tree with `tree.add_file(model, settings)` instead (which
registers a bounding box per instance up front, rather than only
triangulated shapes) fixed it. `select_box(point, extend=...)` specifically
has no matching C++ overload at all in this build — the point form of
`select_box` only accepts a bare point with no tolerance; use `select()`
(precise) instead for a point + radius query, as done below.


In [9]:
settings = ifcopenshell.geom.settings()
tree = ifcopenshell.geom.tree()
tree.add_file(model, settings)
print(f"tree built from {len(model.by_type('IfcElement'))} elements\n")

# --- Clash detection: do any rails intersect the track-element ballast? ---
rails = model.by_type("IfcRail")
track_elements = model.by_type("IfcTrackElement")
print(f"clashing {len(rails)} IfcRail against {len(track_elements)} IfcTrackElement...")

clashes = tree.clash_intersection_many(
    rails, track_elements,
    tolerance=0.002,  # ignore protrusions under 2mm
    check_all=True,
)
print(f"  {len(clashes)} intersection clash(es)")
for clash in clashes[:5]:
    a, b = clash.a, clash.b
    clash_type = ["protrusion", "pierce", "collision", "clearance"][clash.clash_type]
    print(f"  {a.is_a()} {a.Name!r}  x  {b.is_a()} {b.Name!r}  ({clash_type}, {clash.distance:.4f}m)")

# Collision check (surfaces merely touching or intersecting) is cheaper and
# a common first pass before the more expensive intersection check above.
collisions = tree.clash_collision_many(rails, track_elements, allow_touching=True)
print(f"\n  {len(collisions)} collision clash(es) "
      f"(rails resting on/touching track elements without piercing is expected)")


tree built from 75 elements

clashing 4 IfcRail against 66 IfcTrackElement...
  0 intersection clash(es)

  0 collision clash(es) (rails resting on/touching track elements without piercing is expected)


In [10]:
# --- Spatial selection ---
# Build a quick centroid from every shape's placement, rather than assuming
# (0,0,0) is meaningful for this model.
xs, ys, zs = [], [], []
iterator = ifcopenshell.geom.iterator(settings, model, multiprocessing.cpu_count())
if iterator.initialize():
    while True:
        shape = iterator.get()
        loc = ifcopenshell.util.shape.get_shape_matrix(shape)[:, 3][0:3]
        xs.append(loc[0]); ys.append(loc[1]); zs.append(loc[2])
        if not iterator.next():
            break
centre = (float(sum(xs) / len(xs)), float(sum(ys) / len(ys)), float(sum(zs) / len(zs)))
print(f"approximate model centre: {tuple(round(c, 2) for c in centre)}\n")

# select_box(element) -- bounding-box containment/intersection query.
course = model.by_type("IfcCourse")[0]
box_hits = tree.select_box(course)
print(f"select_box(course) -> {len(box_hits)} element(s) sharing/containing its bounding box:")
for el in box_hits[:10]:
    print(f"  {el.is_a()} {el.Name!r}")

# select(point, extend=radius) -- precise geometry query within a sphere.
nearby = tree.select(centre, extend=10.0)
print(f"\nselect(centre, extend=10m) -> {len(nearby)} element(s) within 10m of the model centre")

# select_ray -- fire straight up from the model centre.
results = tree.select_ray(centre, (0.0, 0.0, 1.0), length=50.0)
print(f"\nvertical ray from centre -> {len(results)} intersection(s), nearest first:")
for result in results[:5]:
    hit = model.by_id(result.instance.id())
    print(f"  {hit.is_a()} at distance {result.distance:.2f}m")


approximate model centre: (24.82, 44.85, 7.5)

select_box(course) -> 42 element(s) sharing/containing its bounding box:
  IfcRail 'rail'
  IfcRail 'rail'
  IfcCourse 'ballastbed'
  IfcTrackElement 'sleeper wood'
  IfcTrackElement 'sleeper wood'
  IfcTrackElement 'sleeper wood'
  IfcTrackElement 'sleeper wood'
  IfcTrackElement 'sleeper wood'
  IfcTrackElement 'sleeper wood'
  IfcTrackElement 'sleeper wood'

select(centre, extend=10m) -> 39 element(s) within 10m of the model centre

vertical ray from centre -> 4 intersection(s), nearest first:
  IfcCourse at distance 0.05m
  IfcTrackElement at distance 0.10m
  IfcRail at distance 0.10m
  IfcRail at distance 0.11m


## Summary

- All five documented geometry-exploration paths ran successfully against
  `Infra-Rail.ifc`: individual `create_shape()`, the bulk iterator (with
  the `include` filter), manual entity-graph parsing, glTF/OBJ
  serialisation, and the geometry tree (clash detection + spatial + ray
  queries).
- 73/75 elements produce geometry (2 have no representation).
- No rail-vs-track-element clashes were found — expected, rails rest on
  ballast rather than piercing it.

**Docs vs. this installed build (`ifcopenshell` 0.8.5) — differences found:**

1. `shape.geometry.materials[i]` has no `.original_name()` or
   `.has_diffuse`; `.has_transparency` is a **method**, not a property;
   `.diffuse` is a `colour` object (`.r()`/`.g()`/`.b()`), not a tuple.
2. `create_shape()` has no `@overload` variants, so its declared return
   type is a broad `Union` — a type checker can't narrow it to
   `TriangulationElement` on its own; an explicit `cast(...)` fixes IDE
   warnings without changing runtime behaviour.
3. `tree.select_box()` only returns correct results (including an
   element's own containment) when the tree is built with
   `tree.add_file(model, settings)`, not `tree.add_element(iterator.get())`
   in a loop — the latter only registers triangulated shapes for
   clash/precise-select, not the per-instance bounding boxes `select_box`
   needs.
4. `tree.select_box(point, extend=...)` has no matching overload at all —
   use `tree.select(point, extend=...)` (precise) for a point + radius
   query instead.
5. Python `entity_instance` objects don't have the C++-style
   `.get_argument(0)` the docs' clash examples use — plain attribute
   access (`element.GlobalId`, `element.Name`) or indexing (`element[0]`)
   is the idiomatic Python equivalent.
6. `tree.select(point, extend=...)`'s Python wrapper only recognises a
   point tuple when every element's type is *exactly* `float` — a tuple
   built from numpy scalars (e.g. sliced out of a `get_shape_matrix()`
   result) silently falls through to the wrong overload and raises
   `TypeError`. Cast with `float(...)` first.
